# A small pilot with outside records: borrowing data from patients who were not randomized

## At a glance

A health plan ran a randomized pilot of its transition-navigation offer, with 560 patients. It also
holds records of 2,440 patients from other hospitals, some offered navigation and some not, with no
randomization. Did navigation raise the pilot patients' 30-day transition score, and can the records
tighten the answer?

Borrowing the records, navigation raised the score of pilot patients by 6.07 points, with a 95%
confidence interval of 5.17 to 6.98. The pilot alone gives 6.68 (5.38 to 7.99), so the records cut
the standard error from 0.67 to 0.46. The data are simulated, so we know the true effect, 6.05, and
both intervals contain it. The main caveat is that the interval is valid for the effect inside a
simple description of how the records differ from the pilot, a description the fit chooses from the
data. It is also valid for the pilot's own effect when that description is rich enough, and the data
cannot check that.


## The question

The plan wants to know whether to roll out navigation, and it has one small randomized pilot to
answer with. In the pilot, patients were offered navigation or usual discharge support by a coin
flip. The records come from hospitals that did not randomize: sicker patients were offered
navigation more often, and their scores run higher or lower than the pilot's for reasons other than
the offer. The quantity to estimate is the average difference in 30-day transition score between
offering navigation and not offering it, among the kind of patients the pilot enrolled.

Pooling everyone would be wrong, because the records mix the offer's effect with who was offered
and with how their hospitals differ. Dropping the records is safe but wasteful. The method here,
adaptive TMLE (a variant of targeted maximum likelihood estimation, a method that corrects a
prediction-based estimate so its interval is valid), learns how the records'
outcomes differ from the pilot's as a function of patient characteristics, in the form of a short
formula such as a fixed offset plus a trend in prior utilization, and removes that difference. After removing that difference, it uses the
records as extra patients for the effect. The pilot's own randomization is the only thing the method trusts
outright.


## The data and the setup

In [1]:
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.linear_model import LinearRegression, LogisticRegression

import cleverly
from cleverly import (
    ATE,
    AdaptiveTMLEMethod,
    CausalStudy,
    CrossFitting,
    ModelSpec,
    PointTreatment,
    Runtime,
    TMLEMethod,
    TrialATE,
    TrialWithExternalData,
)

print("cleverly", cleverly.__version__)

cleverly 0.2.0


## The pilot and the records

In [2]:
def simulate(n, seed):
    """Patients of a hospital network, some in a navigation pilot and some in its records."""
    rng = np.random.default_rng(seed)
    discharge_risk = rng.normal(size=n)
    age = rng.normal(size=n)
    prior_utilization = rng.normal(size=n)
    in_pilot = rng.binomial(1, expit(-1.6 + 0.7 * discharge_risk))
    navigation = np.where(
        in_pilot == 1,
        rng.binomial(1, 0.5, n),  # the pilot drew the offer at random
        rng.binomial(1, expit(0.8 * discharge_risk)),  # in the records, sicker patients got it more
    )
    effect = 5 + 2 * discharge_risk  # the offer helps higher-risk patients more
    records_shift = 3 + 2 * prior_utilization  # records score higher than the pilot would
    score = (
        50
        + effect * navigation
        + 3 * discharge_risk
        + 2 * age
        + (1 - in_pilot) * records_shift
        + rng.normal(0, 8, n)
    )
    return pd.DataFrame(
        {
            "in_pilot": in_pilot,
            "transition_navigation": navigation,
            "transition_score": score,
            "discharge_risk": discharge_risk,
            "age": age,
            "prior_utilization": prior_utilization,
        }
    )


frame = simulate(3000, seed=11)

# The known answers come from a very large simulated network.
large = simulate(2_000_000, seed=99)
truth = {
    "pilot patients": float(5 + 2 * large.loc[large["in_pilot"] == 1, "discharge_risk"].mean()),
    "all patients": 5.0,
}
print(list(frame.columns))
print(frame.groupby(["in_pilot", "transition_navigation"]).size().unstack())
print({name: round(value, 2) for name, value in truth.items()})

['in_pilot', 'transition_navigation', 'transition_score', 'discharge_risk', 'age', 'prior_utilization']
transition_navigation     0     1
in_pilot                         
0                      1237  1203
1                       298   262
{'pilot patients': 6.05, 'all patients': 5.0}


Each row is one patient. `in_pilot` is 1 for the 560 pilot patients and 0 for the 2,440 records, and
the offer split of the pilot (298 usual support, 262 navigation) is close to even because it was
drawn at random. Among the records, 1,203 patients were offered navigation, more often the higher-risk
ones. Higher-risk patients are also more likely to be in the pilot, and the offer helps them more,
so the true effect among pilot patients (6.05) is larger than the effect among all patients (5.00).
The simulated characteristics are centered at zero, so a negative value is below the network
average, and scores are in simulated units.


## Why pooling the rows misleads

In [3]:
treated = frame["transition_navigation"] == 1
pooled_gap = (
    frame.loc[treated, "transition_score"].mean() - frame.loc[~treated, "transition_score"].mean()
)
pilot = frame[frame["in_pilot"] == 1]
pilot_treated = pilot["transition_navigation"] == 1
pilot_gap = (
    pilot.loc[pilot_treated, "transition_score"].mean()
    - pilot.loc[~pilot_treated, "transition_score"].mean()
)
print(f"difference in mean score, all rows:   {pooled_gap:.2f}")
print(f"difference in mean score, pilot only: {pilot_gap:.2f}")
print(f"true effect among pilot patients:     {truth['pilot patients']:.2f}")

difference in mean score, all rows:   7.57
difference in mean score, pilot only: 6.87
true effect among pilot patients:     6.05


Comparing offered with other patients across all rows gives 7.57, well above the true 6.05,
because the records' offered patients are sicker and their hospitals score differently. The pilot
alone gives 6.87, which has no such bias but is rough with only 560 patients. The next sections
keep the pilot's randomization and use the records only after the fit has learned how they differ.


## Tell cleverly which rows are the pilot

In [4]:
covariates = ("discharge_risk", "age", "prior_utilization")
design = TrialWithExternalData(
    outcome="transition_score",
    treatment="transition_navigation",
    adjustment=covariates,
    source="in_pilot",  # 1 on pilot rows, 0 on record rows
)
study = CausalStudy(frame, design=design)
effect = study.identify(TrialATE(population="trial"))
print(effect.summary())

                            Causal question (ate)
Question:      trial effect averaged    Rows:                            3,000
               over the trial           Answered by:  adjusting for covariates
               covariates, E[Y^1 - Y^0
               | S = 1]
Outcome:       transition_score
               (continuous)
Treatment:     transition_navigation
               (levels 0, 1)
Adjusted for:  discharge_risk, age,
               prior_utilization
Conditions the answer rests on
condition                checkable   what it says
------------------------------------------------------------------------------
well-defined treatment   no          Each unit's observed outcome is the
                                     outcome it would have had under the
                                     treatment it actually received. This
                                     needs the treatment to be well defined:
                                     every way of getting the same level has


`TrialWithExternalData` tells `cleverly` the outcome, the treatment, the covariates to adjust
for, and which column marks pilot rows. `TrialATE(population="trial")` asks for the effect among
the pilot's patients. The conditions table lists what the answer rests on: the pilot's offer was
random given the covariates (the setup states this, and the data cannot show it), and every kind of
pilot patient could get either arm (the data can check this partly). Nothing is assumed about the
records. The methods list holds one method, because every other method would treat the pooled rows
as one population.


## Estimate the effect with and without the records

In [5]:
fast = ModelSpec(
    outcome_learner=LinearRegression(),
    treatment_learner=LogisticRegression(max_iter=1000),
)

# Pilot rows alone: the usual estimate for a randomized trial.
alone = (
    CausalStudy(
        pilot,
        design=PointTreatment(
            outcome="transition_score",
            treatment="transition_navigation",
            adjustment=covariates,
        ),
    )
    .identify(ATE())
    .estimate(method=TMLEMethod(models=fast, cross_fitting=CrossFitting(enabled=False)))
)

# Pilot and records together.
borrowed = effect.estimate(method=AdaptiveTMLEMethod(models=fast, runtime=Runtime(random_state=1)))

for label, fit in (("pilot alone", alone), ("pilot with records", borrowed)):
    low, high = fit.estimate.ci
    print(
        f"{label:19s} estimate {fit.estimate.psi:.2f}, standard error "
        f"{fit.estimate.std_error:.2f}, 95% interval {low:.2f} to {high:.2f}"
    )
print(f"true effect among pilot patients: {truth['pilot patients']:.2f}")
print(f"label on the pilot-with-records interval: {borrowed.estimate.support.label}")

pilot alone         estimate 6.68, standard error 0.67, 95% interval 5.38 to 7.99
pilot with records  estimate 6.07, standard error 0.46, 95% interval 5.17 to 6.98
true effect among pilot patients: 6.05
label on the pilot-with-records interval: experimental


The first fit is the usual estimate for a randomized trial, on the pilot rows alone. The second
passes all 3,000 rows to `AdaptiveTMLEMethod`. Both use a linear model for the score and a logistic
model for the offer, so the comparison shows the records' contribution and not a change of models.
The `random_state` fixes how the rows are split into five groups, and each row's prediction comes
from models trained on the other groups.

With the records, the estimate is 6.07 and the standard error falls from 0.67 to 0.46, so the
interval is about a third narrower. The narrowing happens here because the records differ from the pilot by the same amount in both
arms, an offset plus a trend in prior utilization, which the fit can learn. Once that difference is
removed, the records' comparison of offered and other patients adds to the pilot's. A gain is not guaranteed: if the records differed in a way the fit could not describe
simply, the interval would stay near the pilot's width. The method labels every fit experimental, because its interval is for the fit's own description
of the difference, and the data cannot confirm that this description is rich enough.


## Check what the fit used

In [6]:
report = borrowed.support_report()
print(f"pilot rows: {report.n_trial}; record rows: {report.n_external}")
print(f"record rows that were offered navigation: {report.n_external_treated}")
print(f"lowest chance that a record row is a pilot row: {report.external_source_min:.3f}")
fold = borrowed.working_models[0]
effect_terms = len(fold.effect.selected)
records_terms = len(fold.bias.selected)
print(f"terms kept for the effect in fold 0: {effect_terms} of {len(fold.dictionary.phi_keep)}")
print(f"terms kept for the records' difference: {records_terms} of {len(fold.dictionary.psi_keep)}")

pilot rows: 560; record rows: 2440
record rows that were offered navigation: 1203
lowest chance that a record row is a pilot row: 0.014
terms kept for the effect in fold 0: 11 of 22
terms kept for the records' difference: 3 of 44


The record rows include 1,203 patients who were offered navigation, so the fit could learn the
difference in both arms. The lowest estimated chance that a record row could be a pilot row is
0.014. For the effect among pilot patients this number is information only, because the pilot
covers itself. The fit kept 11 of 22 candidate terms for the effect and only 3 of 44 for the
records' difference in the first of the five groups of rows (fold 0), and each group selects its
own terms. A short description of the
difference is what lets the records add information, and `borrowed.working_models` lists every
fold's terms.


## What we can and can't conclude

Among pilot-like patients, navigation raised the 30-day transition score by about 6.07 points, with
a 95% interval of 5.17 to 6.98, and borrowing the records cut the pilot's own standard error by
about a third. The answer rests on three conditions. The pilot's offer was random given the covariates,
every kind of pilot patient could get either arm, and the fit described the records' difference from
the pilot well enough. The data cannot check the first or the third, and the interval covers only
the effect inside the fit's own description of that difference.

The page shows one simulated draw, so it does not show how often such an interval covers the truth.
The result says nothing about patients unlike the pilot's. Simple learners are for trying the workflow. When trial enrollment depends on the covariates, an
interval from misfit learners can miss the effect badly, so report a fit with flexible learners. If
the pilot is the only sound evidence the plan has, report the pilot-alone interval beside this one.


## How we know the method works

A simulation study of this method has a recorded result. It drew 2,800 simulated data sets for each
of 17 settings, in most of them with 1,000 trial rows and 3,000 outside rows, and compared the 95%
interval with the known truth. In each of 16 settings the interval covered its target between 93 and 95
times in 100, and the standard error matched the real spread of the estimates to within about 6%.
The target is the trial's effect where the fit can describe how the outside rows differ, and the
fit's own simple-model version of it where the difference is too complicated for that model. With
no difference between the outside rows and the trial, the fit's error was about two thirds of the
trial-only fit's. The study's controls fail where they should: a pooled comparison that ignores
the difference covered the trial's effect only 85 times in 100.

One setting was run as a stress test and is reported without a pass or a fail. Its outcome is
yes or no, and the outside rows differ from the trial in a curved way that the fit's simple model
cannot describe. The two arms' differences are about one standard error apart, so the fit tends
to leave them out, and the interval covered the trial's effect 88 times in 100. That is the cost of the
narrower interval: no interval that stays valid for every size of difference can be much narrower
than the trial alone's where the records agree, so the narrow interval gives up some validity where
the difference is small.

The study used its own simple models and sample sizes, not this page's, and a
second study on selective enrollment has not run. The label stays experimental for every fit
whatever such studies find, because the data cannot confirm that the fit's description of the
difference is rich enough. The
[study page](../technical-reference/method-evidence/adaptive-tmle-trial.md) lists every row, and the
[technical page](../technical-reference/adaptive-tmle.md) lists the unit tests that check the
formulas on exact finite laws.


## Go deeper

### The effect for all patients

In [7]:
everyone = study.identify(TrialATE(population="pooled"))
pooled_fit = everyone.estimate(
    method=AdaptiveTMLEMethod(models=fast, runtime=Runtime(random_state=1))
)
low, high = pooled_fit.estimate.ci
print(
    f"effect among all patients: {pooled_fit.estimate.psi:.2f}, "
    f"95% interval {low:.2f} to {high:.2f}; true {truth['all patients']:.2f}"
)

effect among all patients: 5.10, 95% interval 4.25 to 5.96; true 5.00


The effect among all patients is 5.10, close to the true 5.00, with a 95% interval of 4.25 to
5.96. It is the effect for all patients only if a pilot patient and a record patient with the same
characteristics would respond to the offer alike, which the data cannot check. It differs from the effect among pilot patients because the offer helps sicker patients more
and the pilot enrolled sicker patients. This question needs every record row to resemble some pilot
patients, because the pilot's effect is carried over to the records' kind of patient. The
lowest chance printed above (0.014) stays above the 0.01 level at which the fit warns, so the fit
raised no warning.


## Where to go next

- [One treatment, one outcome](point-treatment-tmle.ipynb) explains the ordinary estimate used
  above for the pilot alone.
- [A randomized pilot](randomized-pilot.ipynb) shows a trial whose offer chances are recorded.
- [A small trial with external data](../user-guide/trial-with-external-data.md) lists the options
  of this method.
- [Adaptive TMLE](../technical-reference/adaptive-tmle.md) states the quantity estimated, the
  conditions and the evidence.
